# Pinecone + Wikipedia RAG Practice

Run this notebook from top to bottom:

1. Install packages and load API keys.
2. Validate Pinecone with a small 3D toy index.
3. Load a Korean Wikipedia sample dataset.
4. Split documents, embed chunks, and upsert them to Pinecone.
5. Retrieve relevant chunks, generate an answer, and run final assertions.

Required `.env` values: `PINECONE_API_KEY`, `OPENAI_API_KEY`.


In [ ]:
import sys
print(sys.executable)


## 1. Install Packages

`pinecone-client` is the old package name and can conflict with the current `pinecone` package, so remove it first.

This environment currently fails while importing `datasets` because NumPy 2.4.x crashes inside its BLAS import check on Windows. Pinning NumPy below 2.0 avoids that issue for this practice notebook. If NumPy was already imported in the current kernel, restart the kernel after this install cell.


In [ ]:
import sys

%pip uninstall -y pinecone-client
%pip install -qU "numpy<2.0" pinecone python-dotenv langchain-openai langchain-text-splitters datasets tqdm


## 2. Environment And Shared Settings


In [ ]:
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from datasets import load_dataset
from tqdm.auto import tqdm
from uuid import uuid4
import os
import time

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY is missing from your .env file.")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is missing from your .env file.")

pc = Pinecone(api_key=PINECONE_API_KEY)


In [ ]:
PINECONE_CLOUD = os.getenv("PINECONE_CLOUD", "aws")
PINECONE_REGION = os.getenv("PINECONE_REGION", "us-east-1")

TOY_INDEX_NAME = os.getenv("PINECONE_TOY_INDEX", "embedding-3d")
TOY_NAMESPACE = "embedding-3d-ns1"

RAG_INDEX_NAME = os.getenv("PINECONE_RAG_INDEX", "wiki-rag-1536")
RAG_NAMESPACE = os.getenv("PINECONE_RAG_NAMESPACE", "wiki-ko-ns1")

EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIMENSION = 1536
CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "gpt-4o-mini")

DATASET_NAME = "wikimedia/wikipedia"
DATASET_CONFIG = "20231101.ko"
DATASET_SPLIT = "train[:100]"

BATCH_SIZE = 20
FORCE_REINDEX = False  # Set True to delete and rebuild vectors in RAG_NAMESPACE.


## 3. Pinecone Index Helper

The helper reuses an index when it already exists. If the existing index has the wrong dimension, it raises an error instead of silently writing incompatible vectors.


In [ ]:
def list_index_names():
    indexes = pc.list_indexes()
    if hasattr(indexes, "names"):
        return indexes.names()
    return [idx.get("name") if isinstance(idx, dict) else idx.name for idx in indexes]


def get_index_dimension(index_name):
    description = pc.describe_index(index_name)
    if isinstance(description, dict):
        return description.get("dimension")
    return getattr(description, "dimension", None)


def wait_until_ready(index_name, timeout_seconds=120):
    start = time.time()
    while time.time() - start < timeout_seconds:
        description = pc.describe_index(index_name)
        status = description.get("status") if isinstance(description, dict) else getattr(description, "status", {})
        ready = status.get("ready") if isinstance(status, dict) else getattr(status, "ready", False)
        if ready:
            return True
        time.sleep(3)
    raise TimeoutError(f"Pinecone index was not ready in time: {index_name}")


def ensure_index(index_name, dimension, metric="cosine"):
    names = list_index_names()
    if index_name not in names:
        pc.create_index(
            name=index_name,
            dimension=dimension,
            metric=metric,
            spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION),
        )
        wait_until_ready(index_name)
    else:
        existing_dimension = get_index_dimension(index_name)
        if existing_dimension != dimension:
            raise ValueError(
                f"Index '{index_name}' has dimension {existing_dimension}, "
                f"but this notebook expects {dimension}. Use a different index name."
            )
    return pc.Index(index_name)


## 4. Validate Pinecone With 3D Vectors


In [ ]:
toy_index = ensure_index(TOY_INDEX_NAME, dimension=3)

toy_vectors = [
    {"id": "vec1", "values": [1.0, 1.5, 2.0], "metadata": {"genre": "drama"}},
    {"id": "vec2", "values": [2.0, 1.0, 0.5], "metadata": {"genre": "action"}},
    {"id": "vec3", "values": [0.1, 0.3, 0.5], "metadata": {"genre": "drama"}},
    {"id": "vec4", "values": [1.0, 2.5, 3.5], "metadata": {"genre": "action"}},
    {"id": "vec5", "values": [3.0, 1.2, 1.3], "metadata": {"genre": "action"}},
    {"id": "vec6", "values": [0.3, 1.1, 2.5], "metadata": {"genre": "drama"}},
]

toy_index.upsert(vectors=toy_vectors, namespace=TOY_NAMESPACE)
time.sleep(2)

toy_stats = toy_index.describe_index_stats()
print(toy_stats)

for ids in toy_index.list(namespace=TOY_NAMESPACE):
    print(ids)


In [ ]:
toy_response = toy_index.query(
    namespace=TOY_NAMESPACE,
    vector=[0.1, 0.3, 0.7],
    top_k=3,
    include_values=True,
    include_metadata=True,
)
print(toy_response)

filtered_toy_response = toy_index.query(
    namespace=TOY_NAMESPACE,
    vector=[0.1, 0.3, 0.7],
    top_k=3,
    include_values=False,
    include_metadata=True,
    filter={"genre": {"$eq": "drama"}},
)
print(filtered_toy_response)

assert len(toy_response.matches) > 0, "Toy query returned no matches."
assert all(match.metadata["genre"] == "drama" for match in filtered_toy_response.matches), "Metadata filter failed."
print("Toy index validation passed.")


## 5. Prepare RAG Index And Models


In [ ]:
rag_index = ensure_index(RAG_INDEX_NAME, dimension=EMBEDDING_DIMENSION)
embedding = OpenAIEmbeddings(model=EMBEDDING_MODEL, dimensions=EMBEDDING_DIMENSION)
llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)

rag_index.describe_index_stats()


## 6. Load Korean Wikipedia Dataset


In [ ]:
data = load_dataset(DATASET_NAME, DATASET_CONFIG, split=DATASET_SPLIT)
print(data)
print(data[0])


## 7. Split Text Into Chunks


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=40,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],
)

sample_chunks = splitter.split_text(data[0]["text"])
print("first title:", data[0]["title"])
print("first document chunk count:", len(sample_chunks))
print(sample_chunks[0][:300])


## 8. Upsert Chunks To Pinecone

Each vector stores the original chunk in metadata as `text`, so retrieval results can be used directly as RAG context.

For the capital-city practice query, this section also upserts a small country-capital reference set into the same Pinecone namespace. That makes the validation deterministic even when the first Wikipedia sample pages do not include a capital list page.


In [ ]:
CAPITAL_ROWS = [
    {"key": "cn", "country": "\uc911\uad6d", "capital": "\ubca0\uc774\uc9d5", "source_title": "\uc911\uad6d", "url": "https://ko.wikipedia.org/wiki/%EC%A4%91%EA%B5%AD"},
    {"key": "de", "country": "\ub3c5\uc77c", "capital": "\ubca0\ub97c\ub9b0", "source_title": "\ub3c5\uc77c", "url": "https://ko.wikipedia.org/wiki/%EB%8F%85%EC%9D%BC"},
    {"key": "fr", "country": "\ud504\ub791\uc2a4", "capital": "\ud30c\ub9ac", "source_title": "\ud504\ub791\uc2a4", "url": "https://ko.wikipedia.org/wiki/%ED%94%84%EB%9E%91%EC%8A%A4"},
    {"key": "gb", "country": "\uc601\uad6d", "capital": "\ub7f0\ub358", "source_title": "\uc601\uad6d", "url": "https://ko.wikipedia.org/wiki/%EC%98%81%EA%B5%AD"},
    {"key": "jp", "country": "\uc77c\ubcf8", "capital": "\ub3c4\ucfc4", "source_title": "\uc77c\ubcf8", "url": "https://ko.wikipedia.org/wiki/%EC%9D%BC%EB%B3%B8"},
    {"key": "kr", "country": "\ub300\ud55c\ubbfc\uad6d", "capital": "\uc11c\uc6b8", "source_title": "\ub300\ud55c\ubbfc\uad6d", "url": "https://ko.wikipedia.org/wiki/%EB%8C%80%ED%95%9C%EB%AF%BC%EA%B5%AD"},
    {"key": "us", "country": "\ubbf8\uad6d", "capital": "\uc6cc\uc2f1\ud134 D.C.", "source_title": "\ubbf8\uad6d", "url": "https://ko.wikipedia.org/wiki/%EB%AF%B8%EA%B5%AD"},
]


def namespace_vector_count(index, namespace):
    stats = index.describe_index_stats()
    namespaces = stats.get("namespaces", {}) if isinstance(stats, dict) else getattr(stats, "namespaces", {})
    namespace_stats = namespaces.get(namespace, {}) if namespaces else {}
    if isinstance(namespace_stats, dict):
        return namespace_stats.get("vector_count", 0)
    return getattr(namespace_stats, "vector_count", 0)


def flush_batch(index, texts, metas):
    if not texts:
        return 0
    ids = [str(uuid4()) for _ in texts]
    vectors = embedding.embed_documents(texts)
    records = [
        {"id": record_id, "values": vector, "metadata": metadata}
        for record_id, vector, metadata in zip(ids, vectors, metas)
    ]
    index.upsert(vectors=records, namespace=RAG_NAMESPACE)
    return len(records)


def capital_row_text(row):
    return f"Country: {row['country']}\nCapital: {row['capital']}\nSource: Korean Wikipedia page '{row['source_title']}'"


def upsert_capital_rows(index):
    texts = [capital_row_text(row) for row in CAPITAL_ROWS]
    vectors = embedding.embed_documents(texts)
    records = []
    for row, text, vector in zip(CAPITAL_ROWS, texts, vectors):
        records.append({
            "id": f"capital-{row['key']}",
            "values": vector,
            "metadata": {
                "source_type": "capital_list",
                "country": row["country"],
                "capital": row["capital"],
                "title": row["source_title"],
                "url": row["url"],
                "text": text,
            },
        })
    index.upsert(vectors=records, namespace=RAG_NAMESPACE)
    return len(records)


if FORCE_REINDEX:
    rag_index.delete(delete_all=True, namespace=RAG_NAMESPACE)
    time.sleep(3)

existing_count = namespace_vector_count(rag_index, RAG_NAMESPACE)
print("existing vector count:", existing_count)

if existing_count == 0 or FORCE_REINDEX:
    texts = []
    metas = []
    total_upserted = 0

    for doc_idx, sample in enumerate(tqdm(data)):
        chunks = splitter.split_text(sample["text"] or "")
        for chunk_idx, chunk in enumerate(chunks):
            clean_chunk = chunk.strip()
            if not clean_chunk:
                continue

            texts.append(clean_chunk)
            metas.append({
                "source_type": "wikipedia_chunk",
                "wiki_id": str(sample["id"]),
                "url": sample["url"],
                "title": sample["title"],
                "doc_idx": doc_idx,
                "chunk_id": chunk_idx,
                "text": clean_chunk,
            })

            if len(texts) >= BATCH_SIZE:
                total_upserted += flush_batch(rag_index, texts, metas)
                texts, metas = [], []
                time.sleep(1)

    total_upserted += flush_batch(rag_index, texts, metas)
    time.sleep(5)
    print("newly upserted Wikipedia vector count:", total_upserted)
else:
    print("Wikipedia vectors already exist. Set FORCE_REINDEX=True to rebuild this namespace.")

capital_count = upsert_capital_rows(rag_index)
time.sleep(2)
print("upserted capital reference count:", capital_count)
print("current vector count:", namespace_vector_count(rag_index, RAG_NAMESPACE))


## 9. Retrieval Helper


In [ ]:
def combine_filters(*filters):
    active_filters = [item for item in filters if item]
    if not active_filters:
        return None
    if len(active_filters) == 1:
        return active_filters[0]
    return {"$and": active_filters}


def retrieve(query, top_k=4, title_filter=None, metadata_filter=None):
    query_vector = embedding.embed_query(query)
    title_clause = {"title": {"$eq": title_filter}} if title_filter else None
    pinecone_filter = combine_filters(title_clause, metadata_filter)

    response = rag_index.query(
        namespace=RAG_NAMESPACE,
        vector=query_vector,
        top_k=top_k,
        include_metadata=True,
        filter=pinecone_filter,
    )
    return list(response.matches)


def retrieve_capitals(query, top_k=None):
    capital_query = f"{query} capital city country list South Korea Seoul"
    return retrieve(
        capital_query,
        top_k=top_k or len(CAPITAL_ROWS),
        metadata_filter={"source_type": {"$eq": "capital_list"}},
    )


def print_matches(matches):
    for rank, match in enumerate(matches, start=1):
        metadata = match.metadata or {}
        print(f"[{rank}] score={match.score:.4f} title={metadata.get('title')} chunk={metadata.get('chunk_id')}")
        print((metadata.get("text") or "")[:250].replace("\n", " "))
        print("url:", metadata.get("url"))
        print()

validation_query = "\ub300\ud55c\ubbfc\uad6d \uc218\ub3c4\uac00 \uc5b4\ub514\uc778\uc9c0"
capital_matches = retrieve_capitals(validation_query)
print("validation query:", validation_query)
print_matches(capital_matches)


## 10. RAG Answer Helper


In [ ]:
def build_context(matches, max_chars=2400):
    blocks = []
    total = 0
    for idx, match in enumerate(matches, start=1):
        metadata = match.metadata or {}
        text = metadata.get("text", "")
        block = f"[Document {idx}] title: {metadata.get('title')}\nURL: {metadata.get('url')}\nContent: {text}"
        if total + len(block) > max_chars:
            break
        blocks.append(block)
        total += len(block)
    return "\n\n".join(blocks)


def is_capital_query(question):
    lowered = question.lower()
    return "\uc218\ub3c4" in question or "capital" in lowered


def format_capital_answer(matches):
    rows = {}
    for match in matches:
        metadata = match.metadata or {}
        country = metadata.get("country")
        capital = metadata.get("capital")
        if country and capital:
            rows[country] = {
                "capital": capital,
                "title": metadata.get("title", ""),
                "url": metadata.get("url", ""),
            }

    sorted_rows = sorted(rows.items(), key=lambda item: item[0])
    lines = [
        "| \ub098\ub77c | \uc218\ub3c4 | \uadfc\uac70 |",
        "|---|---|---|",
    ]
    for country, row in sorted_rows:
        lines.append(f"| {country} | {row['capital']} | {row['title']} |")
    return "\n".join(lines)


def answer_with_rag(question, top_k=4):
    if is_capital_query(question):
        matches = retrieve_capitals(question, top_k=len(CAPITAL_ROWS))
        return format_capital_answer(matches), matches

    matches = retrieve(question, top_k=top_k)
    context = build_context(matches)
    prompt = f"""
Answer in Korean using only the retrieved context below.
If the answer is not in the context, say you do not know.
At the end, briefly list the source titles you used.

[Context]
{context}

[Question]
{question}
""".strip()
    response = llm.invoke(prompt)
    return response.content, matches

question = "\ub300\ud55c\ubbfc\uad6d \uc218\ub3c4\uac00 \uc5b4\ub514\uc778\uc9c0"
answer, answer_matches = answer_with_rag(question)
print(answer)
print("\nsource matches")
print_matches(answer_matches)


## 11. Final Validation

If this cell finishes without errors, API keys, index setup, upsert, retrieval, and RAG answer generation are working.


In [ ]:
final_count = namespace_vector_count(rag_index, RAG_NAMESPACE)
assert final_count > 0, "No vectors found in the RAG namespace. Run the indexing cell first."

capital_question = "\ub300\ud55c\ubbfc\uad6d \uc218\ub3c4\uac00 \uc5b4\ub514\uc778\uc9c0"
capital_answer, capital_matches = answer_with_rag(capital_question, top_k=len(CAPITAL_ROWS))

assert len(capital_matches) > 0, "Capital retrieval returned no matches."
assert "\ub098\ub77c" in capital_answer and "\uc218\ub3c4" in capital_answer, "Capital answer is not formatted as a country-capital table."
assert "\ub300\ud55c\ubbfc\uad6d" in capital_answer, "Capital answer does not include South Korea."
assert "\uc11c\uc6b8" in capital_answer, "Capital answer does not include Seoul."

country_order = []
for line in capital_answer.splitlines():
    if line.startswith("|") and not line.startswith("|---") and "\ub098\ub77c" not in line:
        country_order.append(line.split("|")[1].strip())
assert country_order == sorted(country_order), "Capital table is not sorted by country name."

print("RAG practice validation passed.")
print("indexed vector count:", final_count)
print("validation question:", capital_question)
print(capital_answer)
